## Imputationen, Joins und Pivot/Melt

In dieser Aufgabe lernst du, wie du fehlende Werte imputierst, mehrere Datensätze zusammenfügst und einen breiten Datensatz in ein langes Format umwandelst und wieder zurückführst.

In der Wissensdatenbank findest du ausführliche Anleitungen zu diesen Themen:

- [Wissensdatenbank: Imputation](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/imputation/)
- [Wissensdatenbank: Joins](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/joins/)
- [Wissensdatenbank: Pivot und Melt](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/pivot_melt/)


In [ ]:
import pandas as pd

df = pd.read_csv("../../data/impute.csv")

df

### Aufgabe 1
Imputiere die fehlenden Werte im Datensatz `impute.csv` sinnvoll.


In [ ]:
# Dein Code hier

In [ ]:
# Um richtig interpolieren zu können, müssen wir den Index sortieren.
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp")
df = df.sort_index()

df_interpoliert = df.interpolate(method="linear")
df_interpoliert

In [ ]:
# Kreuze ohne Punkt sind interpoliert. Die Werte von 11:00 bis 15:00 sind eine Gerade über 5 Stunden,
# also geschätzt und nicht gemessen. Ob das sinnvoll ist, musst du fachlich entscheiden.
# pir bleibt leer (es gibt gar keine Werte), die ersten 2 humidity-Werte auch (davor gibt es keinen Wert).
df_interpoliert["temperature"].plot(style="x", label="interpoliert", legend=True)
df["temperature"].plot(style=".", label="gemessen", legend=True)

### Aufgabe 2
Führe zwei Datensätze (`co2.csv` und `temperature.csv`) horizontal zusammen. Behalte dabei alle Zeilen aus dem ersten Datensatz bei aber nur die Zeilen aus dem zweiten Datensatz, die einen passenden Zeitstempel haben.

**Hinweis:** Die Spalten dürfen in den beiden DataFrames nicht gleich heissen. Benenne sie wenn nötig mit `rename` um.


In [ ]:
# Dein Code hier

In [ ]:
# Laden des Datensatzes
co2_df = pd.read_csv('../../data/co2.csv')
co2_df = co2_df.rename(columns={'value': 'co2'})
co2_df = co2_df.set_index("timestamp")

temperature_df = pd.read_csv('../../data/temperature.csv')
temperature_df = temperature_df.rename(columns={'value': 'temperature'})
temperature_df = temperature_df.set_index("timestamp")

joined_df = co2_df.join(temperature_df, how='left')
# joined_df = temperature_df.join(co2_df, how='right')
joined_df

In [ ]:
# left: Der Ausschnitt beginnt mit dem ersten co2-Wert um 23:00, frühere temperature-Werte fallen weg
joined_df.index = pd.to_datetime(joined_df.index)
joined_df["2013-08-23 21:00":"2013-08-24 04:00"].plot(subplots=True, style="o", title="left")

### Aufgabe 3
Führe zwei verschiedene Datensätze so zusammen, dass alle Zeitpunkte erhalten bleiben.


In [ ]:
# Dein Code hier

In [ ]:
joined_df = co2_df.join(temperature_df, how='outer')
joined_df

In [ ]:
# outer: Die temperature-Werte um 21:00 und 22:00 bleiben erhalten, co2 ist dort leer
joined_df.index = pd.to_datetime(joined_df.index)
joined_df["2013-08-23 21:00":"2013-08-24 04:00"].plot(subplots=True, style="o", title="outer")

### Aufgabe 4
Wandle den breiten Datensatz `room_776_wide.csv` mit `melt` in ein langes Format um.


In [ ]:
# Dein Code hier

In [ ]:
df_wide = pd.read_csv('../../data/room_776_wide.csv')
df_wide.head(3)

In [ ]:
df_long = pd.melt(
    df_wide,
    id_vars='timestamp',
    var_name='variable',
    value_name='value'
)

df_long.sample(3)

### Aufgabe 5
Wandle den langen Datensatz `room_776_long.csv` mit `pivot` zurück in ein breites Format um.


In [ ]:
# Dein Code hier

In [ ]:
df_long = pd.read_csv('../../data/room_776_long.csv')
df_long.sample(3)

In [ ]:
df_wide = df_long.pivot(index='timestamp', columns='variable', values='value')

df_wide.head(3)
